In [55]:
!unzip HoaVietNam2026.zip

Archive:  HoaVietNam2026.zip
replace HoaVietNam2026/test/Cuc/0041.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: HoaVietNam2026/test/Cuc/0041.jpg  
  inflating: HoaVietNam2026/test/Cuc/0042.jpg  
  inflating: HoaVietNam2026/test/Cuc/0043.jpg  
  inflating: HoaVietNam2026/test/Cuc/0044.jpg  
  inflating: HoaVietNam2026/test/Cuc/0045.jpg  
  inflating: HoaVietNam2026/test/Cuc/0046.jpg  
  inflating: HoaVietNam2026/test/Cuc/0047.jpg  
  inflating: HoaVietNam2026/test/Cuc/0048.jpg  
  inflating: HoaVietNam2026/test/Cuc/0049.jpg  
  inflating: HoaVietNam2026/test/Cuc/0050.jpg  
  inflating: HoaVietNam2026/test/Dao/0036.jpg  
  inflating: HoaVietNam2026/test/Dao/0037.jpg  
  inflating: HoaVietNam2026/test/Dao/0038.jpg  
  inflating: HoaVietNam2026/test/Dao/0039.jpg  
  inflating: HoaVietNam2026/test/Dao/0040.jpg  
  inflating: HoaVietNam2026/test/Dao/0041.jpg  
  inflating: HoaVietNam2026/test/Dao/0042.jpg  
  inflating: HoaVietNam2026/test/Dao/0043.jpg  
  inflating: HoaVietNam20

## 0. Cài đặt & Import thư viện

In [56]:
import os
import numpy as np
import cv2
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from itertools import product
import warnings
warnings.filterwarnings('ignore')

TRAIN_DIR = '/content/HoaVietNam2026/train'
TEST_DIR  = '/content/HoaVietNam2026/test'

CLASS_NAMES = ['Cuc', 'Dao', 'Ga', 'Lan', 'Mai', 'Sen', 'Tho']

## 1. Hàm trích xuất đặc trưng

### 1a. RGB Histogram (256 bin mỗi kênh)
- Tách ảnh thành 3 kênh R, G, B
- Tính histogram 256 bin cho từng kênh → vector 768 chiều
- Chuẩn hóa để tổng = 1 (normalize)


In [57]:
def extract_rgb_histogram(image_path, bins=256):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    features = []
    for channel in range(3):  # R=0, G=1, B=2
        hist = cv2.calcHist([img], [channel], None, [bins], [0, 256])
        hist = hist.flatten()
        features.extend(hist)

    features = np.array(features, dtype=np.float32)
    features /= (features.sum() + 1e-7)  # Chuẩn hóa
    return features

print(f'Kích thước vector đặc trưng: {3 * 256} chiều')

Kích thước vector đặc trưng: 768 chiều


### 1b. HSV Histogram (8×8×8 bin)

- Chuyển ảnh sang không gian màu HSV (Hue, Saturation, Value)
- 8 bin cho H, 8 bin cho S, 8 bin cho V → vector 512 chiều


In [58]:
def extract_hsv_histogram(image_path, bins=(8, 8, 8)):

    img = cv2.imread(image_path)
    if img is None:
        return None
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Histogram 3D: H trong [0,180], S và V trong [0,256]
    hist = cv2.calcHist(
        [img_hsv], [0, 1, 2], None,
        list(bins),
        [0, 180, 0, 256, 0, 256]
    )
    features = hist.flatten().astype(np.float32)
    features /= (features.sum() + 1e-7)  # Chuẩn hóa
    return features

print(f'Kích thước vector 8x8x8:   {8*8*8} chiều')
print(f'Kích thước vector 16x16x16: {16*16*16} chiều')

Kích thước vector 8x8x8:   512 chiều
Kích thước vector 16x16x16: 4096 chiều


### 1c. Color Moments (Moment màu)

Tính **mean, std, skewness** (trung bình, độ lệch chuẩn, độ lệch phân phối) của 3 kênh HSV → vector 9 chiều.


In [59]:
def extract_color_moments(image_path):
    """
    Tính Color Moments: mean, std, skewness cho 3 kênh HSV
    → vector 9 chiều
    """
    img = cv2.imread(image_path)
    if img is None:
        return None
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    features = []
    for channel in range(3):
        ch = img_hsv[:, :, channel].astype(np.float32)
        mean = np.mean(ch)
        std  = np.std(ch)
        # Skewness (moment bậc 3 chuẩn hóa)
        skew = np.mean(((ch - mean) / (std + 1e-7)) ** 3)
        features.extend([mean, std, skew])

    return np.array(features, dtype=np.float32)

print('Kích thước vector: 9 chiều (mean, std, skew × 3 kênh HSV)')

Kích thước vector: 9 chiều (mean, std, skew × 3 kênh HSV)


### 1d. Dominant Color (Màu chủ đạo)

Dùng **K-Means clustering** để tìm 3 màu chủ đạo trong ảnh → vector 9 chiều.

> 💡 K-Means chia các pixel thành 3 nhóm màu, lấy trung tâm mỗi nhóm làm đại diện.

In [60]:
def extract_dominant_color(image_path, k=3):

    img = cv2.imread(image_path)
    if img is None:
        return None
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Reshape về danh sách pixels (N, 3)
    pixels = img_hsv.reshape(-1, 3).astype(np.float32)

    # K-Means
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, labels, centers = cv2.kmeans(
        pixels, k, None, criteria, 3, cv2.KMEANS_RANDOM_CENTERS
    )

    # Sắp xếp theo tần suất xuất hiện (màu phổ biến nhất trước)
    counts = np.bincount(labels.flatten())
    sorted_idx = np.argsort(-counts)
    centers = centers[sorted_idx]

    return centers.flatten()

print('   Kích thước vector: 9 chiều (3 màu chủ đạo × 3 kênh HSV)')

   Kích thước vector: 9 chiều (3 màu chủ đạo × 3 kênh HSV)


##  2. Hàm tải dữ liệu

Đọc toàn bộ ảnh trong thư mục train/test, trích xuất đặc trưng, gắn nhãn.

In [61]:
def load_dataset(data_dir, extractor_fn, class_names=CLASS_NAMES):

    X, y, paths = [], [], []

    for label_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)

        image_files = [f for f in os.listdir(class_dir)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for img_file in image_files:
            img_path = os.path.join(class_dir, img_file)
            features = extractor_fn(img_path)
            if features is not None:
                X.append(features)
                y.append(label_idx)
                paths.append(img_path)

        print(f'  [{class_name}] {len(image_files)} ảnh')

    return np.array(X), np.array(y), paths


---
# Yêu Cầu 1 — Tìm Tham Số KNN Tốt Nhất (RGB Histogram)

**Mục tiêu:** Grid Search trên các tham số:
- n_neighbors: 1 → 10
- weights: 'uniform' hoặc 'distance'
- metric: 6 loại khoảng cách
.

In [62]:
# Tải dữ liệu với đặc trưng RGB Histogram
print(' Đang tải dữ liệu TRAIN (RGB Histogram 256 bin)...')
X_train_rgb, y_train, _ = load_dataset(TRAIN_DIR, extract_rgb_histogram)

print(f'\n Đang tải dữ liệu TEST (RGB Histogram 256 bin)...')
X_test_rgb, y_test, test_paths = load_dataset(TEST_DIR, extract_rgb_histogram)

print(f'\n Train: {X_train_rgb.shape[0]} ảnh, vector {X_train_rgb.shape[1]} chiều')
print(f' Test : {X_test_rgb.shape[0]} ảnh, vector {X_test_rgb.shape[1]} chiều')

 Đang tải dữ liệu TRAIN (RGB Histogram 256 bin)...
  [Cuc] 40 ảnh
  [Dao] 35 ảnh
  [Ga] 40 ảnh
  [Lan] 35 ảnh
  [Mai] 30 ảnh
  [Sen] 40 ảnh
  [Tho] 30 ảnh

 Đang tải dữ liệu TEST (RGB Histogram 256 bin)...
  [Cuc] 10 ảnh
  [Dao] 12 ảnh
  [Ga] 15 ảnh
  [Lan] 10 ảnh
  [Mai] 10 ảnh
  [Sen] 10 ảnh
  [Tho] 10 ảnh

 Train: 250 ảnh, vector 768 chiều
 Test : 77 ảnh, vector 768 chiều


In [63]:
# Grid Search tìm tham số tốt nhất
n_neighbors_options = list(range(1, 11))          # 1 đến 10
weights_options     = ['uniform', 'distance']
metric_options      = ['braycurtis', 'canberra', 'correlation',
                       'cosine', 'euclidean', 'minkowski']

best_acc    = 0
best_params = {}
results     = []

total = len(n_neighbors_options) * len(weights_options) * len(metric_options)
print(f'🔍 Tổng số tổ hợp cần thử: {total}')
print('   Đang chạy...')

for k, w, m in product(n_neighbors_options, weights_options, metric_options):
    try:
        knn = KNeighborsClassifier(n_neighbors=k, weights=w, metric=m)
        knn.fit(X_train_rgb, y_train)
        y_pred = knn.predict(X_test_rgb)
        acc = accuracy_score(y_test, y_pred)
        results.append({'k': k, 'weights': w, 'metric': m, 'accuracy': acc})

        if acc > best_acc:
            best_acc    = acc
            best_params = {'k': k, 'weights': w, 'metric': m}
    except Exception as e:
        pass

print(f'\n KẾT QUẢ TỐT NHẤT (RGB Histogram):')
print(f'   n_neighbors = {best_params["k"]}')
print(f'   weights     = {best_params["weights"]}')
print(f'   metric      = {best_params["metric"]}')
print(f'   Accuracy    = {best_acc:.4f} ({best_acc*100:.2f}%)')

🔍 Tổng số tổ hợp cần thử: 120
   Đang chạy...

 KẾT QUẢ TỐT NHẤT (RGB Histogram):
   n_neighbors = 3
   weights     = distance
   metric      = braycurtis
   Accuracy    = 0.6883 (68.83%)


In [64]:
# In top 10 kết quả tốt nhất
import pandas as pd

df_results = pd.DataFrame(results).sort_values('accuracy', ascending=False)
print('Top 10 kết quả cao nhất:')
print(df_results.head(10).to_string(index=False))

Top 10 kết quả cao nhất:
 k  weights     metric  accuracy
 3 distance braycurtis  0.688312
10  uniform braycurtis  0.675325
 9  uniform braycurtis  0.675325
 1  uniform   canberra  0.662338
 4 distance braycurtis  0.662338
 5 distance braycurtis  0.662338
 2 distance   canberra  0.662338
 1 distance   canberra  0.662338
 7  uniform braycurtis  0.649351
10 distance braycurtis  0.649351


---
# 🎯 Yêu Cầu 2 — So Sánh Các Loại Đặc Trưng Màu Sắc

Dùng tham số tốt nhất từ Yêu cầu 1, thử nghiệm với 4 loại đặc trưng:

| Đặc trưng | Mô tả | Số chiều |
|-----------|-------|----------|
| RGB Histogram 256 bin | Phân bố pixel theo mức sáng | 768 |
| HSV Histogram 8×8×8 | Phân bố màu trong không gian HSV | 512 |
| Color Moments | Mean, Std, Skew của 3 kênh HSV | 9 |
| Dominant Color | 3 màu chủ đạo (K-Means) | 9 |

In [65]:
# Lấy tham số tốt nhất từ Yêu cầu 1
BEST_K       = best_params['k']
BEST_WEIGHTS = best_params['weights']
BEST_METRIC  = best_params['metric']

print(f' Tham số tối ưu: k={BEST_K}, weights={BEST_WEIGHTS}, metric={BEST_METRIC}')

# Định nghĩa 4 đặc trưng cần so sánh
feature_configs = [
    ('RGB Histogram (256 bin)',   extract_rgb_histogram),
    ('HSV Histogram (8x8x8)',     lambda p: extract_hsv_histogram(p, bins=(8,8,8))),
    ('Color Moments',             extract_color_moments),
    ('Dominant Color (K-Means)',  extract_dominant_color),
]

comparison_results = []

for feat_name, extractor in feature_configs:
    print(f'\n Đang xử lý: {feat_name} ...')

    X_tr, y_tr, _           = load_dataset(TRAIN_DIR, extractor)
    X_te, y_te, test_paths_ = load_dataset(TEST_DIR,  extractor)

    try:
        knn = KNeighborsClassifier(
            n_neighbors=BEST_K,
            weights=BEST_WEIGHTS,
            metric=BEST_METRIC
        )
        knn.fit(X_tr, y_tr)
        y_pred = knn.predict(X_te)
        acc = accuracy_score(y_te, y_pred)
    except Exception:
        # Fallback về euclidean nếu metric không tương thích
        knn = KNeighborsClassifier(n_neighbors=BEST_K, metric='euclidean')
        knn.fit(X_tr, y_tr)
        y_pred = knn.predict(X_te)
        acc = accuracy_score(y_te, y_pred)

    comparison_results.append({'Đặc trưng': feat_name, 'Accuracy': acc})
    print(f' Accuracy: {acc:.4f} ({acc*100:.2f}%)')

print('\n BẢNG SO SÁNH:')
df_comp = pd.DataFrame(comparison_results).sort_values('Accuracy', ascending=False)
print(df_comp.to_string(index=False))

 Tham số tối ưu: k=3, weights=distance, metric=braycurtis

 Đang xử lý: RGB Histogram (256 bin) ...
  [Cuc] 40 ảnh
  [Dao] 35 ảnh
  [Ga] 40 ảnh
  [Lan] 35 ảnh
  [Mai] 30 ảnh
  [Sen] 40 ảnh
  [Tho] 30 ảnh
  [Cuc] 10 ảnh
  [Dao] 12 ảnh
  [Ga] 15 ảnh
  [Lan] 10 ảnh
  [Mai] 10 ảnh
  [Sen] 10 ảnh
  [Tho] 10 ảnh
 Accuracy: 0.6883 (68.83%)

 Đang xử lý: HSV Histogram (8x8x8) ...
  [Cuc] 40 ảnh
  [Dao] 35 ảnh
  [Ga] 40 ảnh
  [Lan] 35 ảnh
  [Mai] 30 ảnh
  [Sen] 40 ảnh
  [Tho] 30 ảnh
  [Cuc] 10 ảnh
  [Dao] 12 ảnh
  [Ga] 15 ảnh
  [Lan] 10 ảnh
  [Mai] 10 ảnh
  [Sen] 10 ảnh
  [Tho] 10 ảnh
 Accuracy: 0.7662 (76.62%)

 Đang xử lý: Color Moments ...
  [Cuc] 40 ảnh
  [Dao] 35 ảnh
  [Ga] 40 ảnh
  [Lan] 35 ảnh
  [Mai] 30 ảnh
  [Sen] 40 ảnh
  [Tho] 30 ảnh
  [Cuc] 10 ảnh
  [Dao] 12 ảnh
  [Ga] 15 ảnh
  [Lan] 10 ảnh
  [Mai] 10 ảnh
  [Sen] 10 ảnh
  [Tho] 10 ảnh
 Accuracy: 0.4935 (49.35%)

 Đang xử lý: Dominant Color (K-Means) ...
  [Cuc] 40 ảnh
  [Dao] 35 ảnh
  [Ga] 40 ảnh
  [Lan] 35 ảnh
  [Mai] 30 ảnh
  [Sen

---
# 🎯 Yêu Cầu 3 — Phân Tích Trường Hợp Sai/Đúng (A vs B)

**A = RGB Histogram 256 bin**  
**B = HSV Histogram 16×16×16**

So sánh từng ảnh test:
- **A sai, B đúng** → B giỏi hơn ở những mẫu này
- **A đúng, B sai** → A giỏi hơn ở những mẫu này  
- **A sai và B sai** → Mẫu khó, cả hai đều thất bại

In [66]:
# Huấn luyện mô hình A (RGB Histogram 256 bin)
print(' Tải dữ liệu cho mô hình A ')
X_train_A, y_train_A, _ = load_dataset(TRAIN_DIR, extract_rgb_histogram)
X_test_A,  y_test_A, test_paths_A = load_dataset(TEST_DIR, extract_rgb_histogram)

knn_A = KNeighborsClassifier(
    n_neighbors=BEST_K, weights=BEST_WEIGHTS, metric=BEST_METRIC
)
try:
    knn_A.fit(X_train_A, y_train_A)
    pred_A = knn_A.predict(X_test_A)
except:
    knn_A = KNeighborsClassifier(n_neighbors=BEST_K, metric='euclidean')
    knn_A.fit(X_train_A, y_train_A)
    pred_A = knn_A.predict(X_test_A)

print(f' Mô hình A - Accuracy: {accuracy_score(y_test_A, pred_A):.4f}')

# Huấn luyện mô hình B (HSV Histogram 16×16×16)
print('\n Tải dữ liệu cho mô hình B ')
extractor_B = lambda p: extract_hsv_histogram(p, bins=(16, 16, 16))
X_train_B, y_train_B, _ = load_dataset(TRAIN_DIR, extractor_B)
X_test_B,  y_test_B, test_paths_B = load_dataset(TEST_DIR, extractor_B)

knn_B = KNeighborsClassifier(
    n_neighbors=BEST_K, weights=BEST_WEIGHTS, metric=BEST_METRIC
)
try:
    knn_B.fit(X_train_B, y_train_B)
    pred_B = knn_B.predict(X_test_B)
except:
    knn_B = KNeighborsClassifier(n_neighbors=BEST_K, metric='euclidean')
    knn_B.fit(X_train_B, y_train_B)
    pred_B = knn_B.predict(X_test_B)

print(f' Mô hình B - Accuracy: {accuracy_score(y_test_B, pred_B):.4f}')

 Tải dữ liệu cho mô hình A 
  [Cuc] 40 ảnh
  [Dao] 35 ảnh
  [Ga] 40 ảnh
  [Lan] 35 ảnh
  [Mai] 30 ảnh
  [Sen] 40 ảnh
  [Tho] 30 ảnh
  [Cuc] 10 ảnh
  [Dao] 12 ảnh
  [Ga] 15 ảnh
  [Lan] 10 ảnh
  [Mai] 10 ảnh
  [Sen] 10 ảnh
  [Tho] 10 ảnh
 Mô hình A - Accuracy: 0.6883

 Tải dữ liệu cho mô hình B 
  [Cuc] 40 ảnh
  [Dao] 35 ảnh
  [Ga] 40 ảnh
  [Lan] 35 ảnh
  [Mai] 30 ảnh
  [Sen] 40 ảnh
  [Tho] 30 ảnh
  [Cuc] 10 ảnh
  [Dao] 12 ảnh
  [Ga] 15 ảnh
  [Lan] 10 ảnh
  [Mai] 10 ảnh
  [Sen] 10 ảnh
  [Tho] 10 ảnh
 Mô hình B - Accuracy: 0.8052


In [70]:
# Phân tích từng ảnh test
# Đảm bảo y_test từ hai extractor là cùng thứ tự ảnh

cases = {
    'A_sai_B_dung': [],
    'A_dung_B_sai': [],
    'A_sai_B_sai':  [],
}

n = min(len(pred_A), len(pred_B))
for i in range(n):
    true_label  = y_test_A[i]
    a_correct   = (pred_A[i] == true_label)
    b_correct   = (pred_B[i] == true_label)

    info = {
        'path':       test_paths_A[i],
        'true':       CLASS_NAMES[true_label],
        'pred_A':     CLASS_NAMES[pred_A[i]],
        'pred_B':     CLASS_NAMES[pred_B[i]],
    }

    if not a_correct and b_correct:
        cases['A_sai_B_dung'].append(info)
    elif a_correct and not b_correct:
        cases['A_dung_B_sai'].append(info)
    elif not a_correct and not b_correct:
        cases['A_sai_B_sai'].append(info)

print('=' * 60)
print(f"  PHÂN TÍCH KẾT QUẢ (trên {n} ảnh test):")
print(f"  A sai, B đúng  (ưu điểm B): {len(cases['A_sai_B_dung'])} ảnh")
print(f"  A đúng, B sai  (ưu điểm A): {len(cases['A_dung_B_sai'])} ảnh")
print(f"  A sai và B sai (mẫu khó)  : {len(cases['A_sai_B_sai'])} ảnh")
print('=' * 60)

  PHÂN TÍCH KẾT QUẢ (trên 77 ảnh test):
  A sai, B đúng  (ưu điểm B): 11 ảnh
  A đúng, B sai  (ưu điểm A): 2 ảnh
  A sai và B sai (mẫu khó)  : 13 ảnh


In [69]:
def print_cases(case_list, title, max_show=10):
    print(f'\n{"=" * 60}')
    print(f' {title} ({len(case_list)} ảnh):')
    print(f'{"=" * 60}')
    for item in case_list[:max_show]:
        filename = os.path.basename(item['path'])
        folder   = os.path.basename(os.path.dirname(item['path']))
        print(f"    {folder}/{filename}")
        print(f"     Nhãn thật: {item['true']}  |  A dự đoán: {item['pred_A']}  |  B dự đoán: {item['pred_B']}")
    if len(case_list) > max_show:
        print(f'  ... và {len(case_list) - max_show} ảnh khác')

print_cases(cases['A_sai_B_dung'], 'A SAI - B ĐÚNG → Ưu điểm của B (HSV 16x16x16)')
print_cases(cases['A_dung_B_sai'], 'A ĐÚNG - B SAI → Ưu điểm của A (RGB 256 bin)')
print_cases(cases['A_sai_B_sai'],  'A SAI và B SAI → Mẫu khó, cả hai thất bại')


 A SAI - B ĐÚNG → Ưu điểm của B (HSV 16x16x16) (11 ảnh):
    Ga/0052.jpg
     Nhãn thật: Ga  |  A dự đoán: Tho  |  B dự đoán: Ga
    Ga/0047.jpg
     Nhãn thật: Ga  |  A dự đoán: Lan  |  B dự đoán: Ga
    Ga/0051.jpg
     Nhãn thật: Ga  |  A dự đoán: Dao  |  B dự đoán: Ga
    Lan/0045.jpg
     Nhãn thật: Lan  |  A dự đoán: Dao  |  B dự đoán: Lan
    Mai/0035.jpg
     Nhãn thật: Mai  |  A dự đoán: Tho  |  B dự đoán: Mai
    Mai/0034.jpg
     Nhãn thật: Mai  |  A dự đoán: Tho  |  B dự đoán: Mai
    Mai/0039.jpg
     Nhãn thật: Mai  |  A dự đoán: Tho  |  B dự đoán: Mai
    Mai/0040.jpg
     Nhãn thật: Mai  |  A dự đoán: Tho  |  B dự đoán: Mai
    Sen/0045.jpg
     Nhãn thật: Sen  |  A dự đoán: Cuc  |  B dự đoán: Sen
    Sen/0046.jpg
     Nhãn thật: Sen  |  A dự đoán: Dao  |  B dự đoán: Sen
  ... và 1 ảnh khác

 A ĐÚNG - B SAI → Ưu điểm của A (RGB 256 bin) (2 ảnh):
    Dao/0041.jpg
     Nhãn thật: Dao  |  A dự đoán: Dao  |  B dự đoán: Mai
    Mai/0036.jpg
     Nhãn thật: Mai  |  A dự đoán